# Fast spot quantification — per-bit foci intensity QC

Bit-to-bit hybridization reagent quality is not uniform: a compromised bit's foci can look visibly dimmer/sparser than its neighbours, and catching that early (while the round could still be re-imaged/re-hybridized) is much cheaper than finding out after the whole experiment is decoded.

This notebook takes a small set of representative FOVs (`SELECTED_FOVS`) and, for every resolved (round, color) combination, measures each detected focus's background-subtracted peak intensity:

1. For each color, sample `N_Z_SAMPLES` z-planes (evenly spaced across that round's real z-sweep for the color) and read just those frames.
2. Crop each frame to a `CROP_SIZE`x`CROP_SIZE` square centered on the FOV, to avoid illumination vignetting near the frame edges biasing the comparison.
3. Max-project the sampled z-planes, then detect foci via `MERci.analysis.spot_localization.detect_beads_2d` (Gaussian-smoothed background estimate + local-maxima threshold — the same "spot recognition after background subtraction" primitive this repo already uses for bead/fiducial detection, reused here for real hybridization foci instead).
4. Record each detected focus's peak pixel value minus the crop's own background median.

Results are cached to `SAMPLE_DIR/analysis/fast_spot_quantification/` as one CSV per (round, color, FOV) — re-running the notebook (e.g. as more rounds/hybs are imaged) only computes new combinations; set `OVERWRITE_OUTPUT = True` to force recomputation.

The final plot pools every `SELECTED_FOVS` FOV's foci together per round: one row per real bit color, x = round number, one semi-transparent violin per round showing that round's spot-intensity distribution for that color — a round whose reagent degraded should stand out as a visibly lower/tighter violin among its neighbours.

This is a fast relative-QC signal, not a calibrated molecule-counting pipeline — `FOCI_MIN_DIST_PX`/`FOCI_THRESH_SIGMA` (Section 3) are tuned by eye against the diagnostic overlay (Section 4), not against ground truth.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.acquisition.configs    import find_frame_table_for_hal_config, get_all_color_frame_indices
from MERci.analysis.spot_localization import detect_beads_2d
from MERci.progress_display       import ProgressReporter

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "fast_spot_quantification"   # namespaces this notebook's figures
MICROSCOPE    = "MF3"

# Which FOVs to sample -- a handful of representative FOVs is enough for a
# fast relative QC signal; these are global fov_id values, same convention as
# every other notebook/positions file in this repo.
SELECTED_FOVS = [0, 1, 2]

# Square crop side length (pixels), centered on the raw frame -- restricts
# every measurement to the FOV's own center, away from the illumination
# vignetting that gets worse toward the frame edges, so intensities are
# comparable round-to-round without needing flat-field correction here.
CROP_SIZE = 400

# How many z-planes to sample per (round, color), evenly spaced across that
# round's own real z-sweep for the color (see Section 3) -- these are
# max-projected before spot detection (Section 4), so a focus slightly out
# of the plane closest to its true z is still caught. A single frame
# (N_Z_SAMPLES=1) is fastest but will miss out-of-focus foci; the round's
# full z-stack is most thorough but reads more data per FOV -- tune this
# against how variable your tissue's real focus depth is.
N_Z_SAMPLES = 5

# Colors to exclude from every round's resolution (Section 3) -- 405 nm is
# the cells/DAPI round's own channel (not a hybridization bit), 488 nm is
# HAL's bead/focus-lock reference channel (always at a fixed bead z, never
# real tissue signal). Real bit colors (typically 560/650/750) are picked up
# automatically from each round's own frame table.
EXCLUDED_COLORS = [405.0, 488.0]

# Foci-detection parameters for detect_beads_2d (Section 4) -- tuned by eye
# against the diagnostic overlay in Section 5, not calibrated against ground
# truth. FOCI_MIN_DIST_PX is the minimum center-to-center spacing (pixels)
# between accepted foci; FOCI_THRESH_SIGMA is the detection threshold above
# the estimated background, in units of the background's own std.
FOCI_MIN_DIST_PX  = 4
FOCI_THRESH_SIGMA = 3.0

# False (default): skip recomputing any (round, color, FOV) combination that
# already has a cached CSV in OUTPUT_DIR (Section 6) -- lets this notebook be
# re-run cheaply as new rounds/hybs are imaged. True: recompute and overwrite
# every combination regardless of what's already cached.
OVERWRITE_OUTPUT = False

# Shared plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- reused by every
# plotting cell in this notebook.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 10
PLOT_LEGEND_FONTSIZE = 10

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# Per-experiment output data (explicitly requested location, distinct from
# NOTEBOOK_GUIDELINES.md #2's generic analysis/cache/<notebook_name>/ -- this
# is real quantification output meant to persist, like analysis/thumbnails/
# or analysis/mosaics/, not just an internal recompute cache).
OUTPUT_DIR = config.analysis_dir / "fast_spot_quantification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = config.analysis_dir / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name    : {SAMPLE_NAME}")
print(f"FOVs (total)   : {meta.n_fovs}")
print(f"Rounds         : {sorted(meta.rounds)}")
print(f"SELECTED_FOVS  : {SELECTED_FOVS}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")

## 3 — Resolve each round's real bit colors -> sampled z-frame indices

Mirrors `round_mosaics.ipynb`'s own frame-table resolution (Section 3 there), but instead of the single frame closest to a target z, keeps `N_Z_SAMPLES` frame indices evenly spaced across `get_all_color_frame_indices`' full ascending-z list for that color -- these get max-projected before spot detection (Section 4).

In [ ]:
def sample_z_frame_indices(all_indices, n_samples):
    # Evenly spaced picks from all_indices (already ascending z order) -- if
    # there are fewer real z-planes than requested, just use all of them.
    if len(all_indices) <= n_samples:
        return list(all_indices)
    positions = np.linspace(0, len(all_indices) - 1, n_samples)
    picked = sorted({all_indices[int(round(p))] for p in positions})
    return picked


def resolve_round_color_frame_indices(round_id):
    # {color_nm: [frame_idx, ...]} for round_id's own frame table -- every
    # real color it has (except EXCLUDED_COLORS and blanks, which have no
    # color at all), each with N_Z_SAMPLES sampled frame indices.
    color_frames = {}
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        frame_table_path = find_frame_table_for_hal_config(
            config.settings_dir / s.hal_config, config.metadata_dir)
        if frame_table_path is None:
            continue
        frame_table = pd.read_csv(frame_table_path)
        for color in sorted(frame_table["color"].dropna().unique()):
            if any(round(color) == round(excluded) for excluded in EXCLUDED_COLORS):
                continue
            all_indices = get_all_color_frame_indices(frame_table, float(color))
            if not all_indices:
                continue
            color_frames[float(color)] = sample_z_frame_indices(all_indices, N_Z_SAMPLES)
    return color_frames


ROUND_COLOR_FRAME_INDICES = {}
for round_id in sorted(meta.rounds):
    cf = resolve_round_color_frame_indices(round_id)
    if cf:
        ROUND_COLOR_FRAME_INDICES[round_id] = cf
        n_samples_per_color = {c: len(f) for c, f in cf.items()}
        print(f"Round {round_id}: colors {sorted(cf)} -> n_z_samples per color: {n_samples_per_color}")
    else:
        print(f"Round {round_id}: no bit colors resolved")

## 4 — Foci detection helpers

`detect_foci_in_crop` is the shared core: crop every sampled z-frame to `CROP_SIZE`, max-project, estimate a background median (same Gaussian-smoothed, bottom-80th-percentile convention `detect_beads_2d` itself uses internally for its detection threshold, kept consistent here so the *subtracted* background matches the *detection* background), then detect candidate foci via `detect_beads_2d`. Used by both the diagnostic overlay (Section 5) and the real per-combination computation (Section 6).

In [ ]:
def crop_center(frame, crop_size):
    h, w = frame.shape
    y0 = max(0, h // 2 - crop_size // 2)
    x0 = max(0, w // 2 - crop_size // 2)
    y1 = min(h, y0 + crop_size)
    x1 = min(w, x0 + crop_size)
    return frame[y0:y1, x0:x1]


def compute_background_median(image, bg_percentile=80):
    blurred = gaussian_filter(image.astype(float), sigma=1.5)
    bg_mask = blurred < np.percentile(blurred, bg_percentile)
    return float(np.median(blurred[bg_mask])) if bg_mask.any() else float(np.median(blurred))


def detect_foci_in_crop(frames, crop_size, min_dist_px, thresh_sigma):
    # frames: (n_z, H, W) raw array for one (fov, round, color). Returns
    # (max_proj, bg_med, candidates) -- candidates is the (N, 2) [row, col]
    # array detect_beads_2d returns.
    cropped  = np.stack([crop_center(f, crop_size) for f in frames], axis=0)
    max_proj = cropped.max(axis=0).astype(np.float32)
    bg_med   = compute_background_median(max_proj)
    candidates = detect_beads_2d(max_proj, min_dist_px, thresh_sigma)
    return max_proj, bg_med, candidates


def spot_cache_path(round_id, color_nm, fov_id):
    return OUTPUT_DIR / f"round{round_id:03d}_{color_nm:.0f}nm_fov{fov_id:04d}.csv"


def compute_fov_round_color_spots(fov_id, round_id, color_nm, frame_indices, series):
    # Returns a DataFrame of detected foci, or None if fov_id isn't imaged
    # yet for this round (so it can be retried on a later run rather than
    # cached as an empty/wrong result).
    existing = [p for p in (s.resolve_path(fov_id, config.image_suffix) for s in series) if p.exists()]
    if not existing:
        return None
    frames = read_image_frames(existing[0], frame_indices,
                                frame_width=config.frame_width, frame_height=config.frame_height)
    max_proj, bg_med, candidates = detect_foci_in_crop(frames, CROP_SIZE, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)
    rows = [
        {"fov": fov_id, "round": round_id, "color_nm": color_nm,
         "row_px": int(r), "col_px": int(c), "intensity": float(max_proj[r, c] - bg_med)}
        for (r, c) in candidates
    ]
    return pd.DataFrame(rows, columns=["fov", "round", "color_nm", "row_px", "col_px", "intensity"])

## 5 — Diagnostic: visualize detected foci on one example crop

Before running the full batch below, sanity-check `FOCI_MIN_DIST_PX`/`FOCI_THRESH_SIGMA` (Section 2) against one real, already-imaged (round, color, FOV) combination -- re-run this cell after adjusting either parameter. Picks the first resolved combination that has at least one `SELECTED_FOVS` entry already imaged; prints a message instead if nothing is imaged yet.

In [ ]:
EXAMPLE = None
for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items():
    series = meta.series_for_round(round_id)
    for color_nm, frame_indices in color_frames.items():
        for fov_id in SELECTED_FOVS:
            existing = [p for p in (s.resolve_path(fov_id, config.image_suffix) for s in series) if p.exists()]
            if existing:
                EXAMPLE = (round_id, color_nm, fov_id, frame_indices, existing[0])
                break
        if EXAMPLE:
            break
    if EXAMPLE:
        break

if EXAMPLE is None:
    print("No imaged (round, color, FOV) combination found yet for the diagnostic overlay -- "
          "run again once imaging has started.")
else:
    example_round_id, example_color_nm, example_fov_id, example_frame_indices, example_path = EXAMPLE
    frames = read_image_frames(example_path, example_frame_indices,
                                frame_width=config.frame_width, frame_height=config.frame_height)
    max_proj, bg_med, candidates = detect_foci_in_crop(frames, CROP_SIZE, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)

    vmin, vmax = np.percentile(max_proj, [1.0, 99.0])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(max_proj, cmap="gray", vmin=vmin, vmax=vmax)
    if len(candidates):
        ax.scatter(candidates[:, 1], candidates[:, 0], s=40, facecolors="none",
                   edgecolors="red", linewidths=1.0)
    ax.set_title(f"round {example_round_id}, {example_color_nm:.0f} nm, FOV {example_fov_id} "
                 f"-- {len(candidates)} foci detected", fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.example_foci_overlay.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")

## 6 — Run the quantification

For every (round, color, FOV) combination resolved above, skips any that already has a cached CSV in `OUTPUT_DIR` unless `OVERWRITE_OUTPUT=True` -- so re-running this notebook as more rounds/hybs get imaged only computes what's new. A combination whose FOV isn't imaged yet is skipped without caching anything, so it's retried on a later run rather than permanently recorded as empty.

In [ ]:
combos = [
    (fov_id, round_id, color_nm, frame_indices)
    for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items()
    for color_nm, frame_indices in color_frames.items()
    for fov_id in SELECTED_FOVS
]

n_computed, n_skipped_cached, n_not_imaged = 0, 0, 0
reporter = ProgressReporter(total=len(combos), label="Quantifying foci")
for fov_id, round_id, color_nm, frame_indices in reporter.wrap(combos):
    cache_path = spot_cache_path(round_id, color_nm, fov_id)
    if cache_path.exists() and not OVERWRITE_OUTPUT:
        n_skipped_cached += 1
        continue

    series = meta.series_for_round(round_id)
    df = compute_fov_round_color_spots(fov_id, round_id, color_nm, frame_indices, series)
    if df is None:
        n_not_imaged += 1
        continue

    df.to_csv(cache_path, index=False)
    n_computed += 1

print(f"Computed: {n_computed} | Skipped (already cached): {n_skipped_cached} | Not yet imaged: {n_not_imaged}")

## 7 — Load cached results

In [ ]:
def load_all_spots():
    frames = []
    for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items():
        for color_nm in color_frames:
            for fov_id in SELECTED_FOVS:
                cache_path = spot_cache_path(round_id, color_nm, fov_id)
                if cache_path.exists():
                    frames.append(pd.read_csv(cache_path))
    if not frames:
        return pd.DataFrame(columns=["fov", "round", "color_nm", "row_px", "col_px", "intensity"])
    return pd.concat(frames, ignore_index=True)


SPOTS_DF = load_all_spots()
n_combos = SPOTS_DF[["fov", "round", "color_nm"]].drop_duplicates().shape[0] if not SPOTS_DF.empty else 0
print(f"Loaded {len(SPOTS_DF)} foci across {n_combos} (fov, round, color) combination(s).")

## 8 — Plot: spot-intensity violin per bit color, across rounds

One row per real bit color found in `SPOTS_DF`; within each row, one semi-transparent (`alpha=0.5`) violin per round, positioned at that round's actual `imaging_round` number on the x-axis, pooling every `SELECTED_FOVS` FOV's foci together. A round whose reagent degraded should show up as a visibly lower and/or tighter violin relative to its neighbours.

In [ ]:
if SPOTS_DF.empty:
    print("No foci data yet -- nothing to plot. Run Section 6 once at least one combination is imaged.")
else:
    colors_present = sorted(SPOTS_DF["color_nm"].unique())
    fig, axes = plt.subplots(len(colors_present), 1, figsize=(10, 4 * len(colors_present)), squeeze=False)

    for ax, color_nm in zip(axes[:, 0], colors_present):
        sub = SPOTS_DF[SPOTS_DF["color_nm"] == color_nm]
        rounds_present = sorted(sub["round"].unique())
        per_round_data = [sub.loc[sub["round"] == r, "intensity"].values for r in rounds_present]
        # violinplot errors on an empty array -- drop rounds with zero detected
        # foci rather than crashing the whole subplot.
        valid = [(r, d) for r, d in zip(rounds_present, per_round_data) if len(d) > 0]
        if valid:
            vp_rounds, vp_data = zip(*valid)
            parts = ax.violinplot(vp_data, positions=vp_rounds, showmedians=True)
            for body in parts["bodies"]:
                body.set_alpha(0.5)
            ax.set_xticks(vp_rounds)
        ax.set_title(f"{color_nm:.0f} nm", fontsize=PLOT_TITLE_FONTSIZE)
        ax.set_xlabel("Round", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("Spot intensity (bg-subtracted)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.spot_intensity_violin.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")